# repovideo — Colab GPU Worker

This notebook installs your **actual repovideo project** from GitHub and runs the GPU-heavy parts on Colab's A100/T4. No copy-pasted code — when you update the repo, the notebook gets the updates too.

### Quick start
1. Run **Setup** (cells 1-2)
2. Pick a path:
   - **Section A** — Train a LoRA on a built-in HF dataset (or your own images)
   - **Section B** — Generate an anecdote video
   - **Section C** — Download results to your Mac
   - **Section D** — Run as automated worker for `--colab` flag

In [ ]:
#@title 1. Install repovideo from your GitHub repo
#@markdown This clones your repo and installs it as a real Python package.
#@markdown Change the URL if you've forked it.

REPO_URL = 'https://github.com/ibrahim-ansari-code/repo---video.git' #@param {type:"string"}

!pip install -q "git+{REPO_URL}[train]"
# decord for fast video loading during training
!pip install -q decord

print('\n✅ repovideo installed')
!repovideo --help

In [ ]:
#@title 2. Check GPU + see available datasets
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'GPU:  {gpu}')
    print(f'VRAM: {vram:.1f} GB')
    RECOMMENDED = '14B' if vram >= 38 else '1.3B'
    print(f'→ Recommended model: {RECOMMENDED}')
else:
    print('⚠️  No GPU. Go to Runtime → Change runtime type → T4 GPU')
    RECOMMENDED = '1.3B'

print()
!repovideo list-datasets

---
## Section A: Train a LoRA

Two options:
- **A1**: Use a built-in HF dataset (no uploads needed)
- **A2**: Upload your own reference videos/images

**Dataset tiers (best → worst for I2V LoRA):**
| Dataset | Tier | What it has | What it trains |
|---------|------|-------------|----------------|
| `tip-i2v` | **I2V** | Image + text prompt pairs (ICCV 2025) | Image-to-video conditioning (**recommended**) |
| `pusa` | VIDEO | Video + caption (Wan-T2V generated) | Motion + style (T2V, no image condition) |
| `cinematic` | VIDEO | Film clips + scene descriptions | Motion + style |
| `pexels` | VIDEO | Stock footage via URLs | Motion + style |
| `developer` | IMAGE | Tech/dev scene thumbnails | Style only (no motion) |
| `dramatic` | IMAGE | Moody/dramatic thumbnails | Style only (no motion) |

In [ ]:
#@title A1: Train on a built-in dataset (recommended to start)
#@markdown **pusa** and **cinematic** are VIDEO tier (best quality).
#@markdown **developer** and **dramatic** are IMAGE-only (style only, no motion).

DATASET = 'tip-i2v' #@param ["tip-i2v", "pusa", "cinematic", "pexels", "developer", "dramatic"]
MODEL_SIZE = '14B' #@param ["14B", "1.3B"]
TRAIN_STEPS = 500 #@param {type:"integer"}
NUM_FRAMES = 17 #@param {type:"integer"}

from src.config import ensure_dirs
from src.anecdote.datasets import download_dataset, get_dataset_tier
from src.anecdote.lora_trainer import LoRATrainingConfig, train_lora

ensure_dirs()

tier = get_dataset_tier(DATASET)
print(f'Dataset: {DATASET} (tier: {tier})')
print(f'Downloading from Hugging Face...')
data_dir = download_dataset(DATASET)

print(f'\nTraining LoRA ({TRAIN_STEPS} steps, Wan2.1 {MODEL_SIZE}, {NUM_FRAMES} frames)...')
if tier == 'image':
    print('⚠️  Image-only dataset: will learn style but NOT motion.')
    print('   Use "pusa" or "cinematic" for full video diffusion training.\n')

config = LoRATrainingConfig(
    reference_dir=data_dir,
    dataset_name=DATASET,
    lora_name=DATASET,
    model_size=MODEL_SIZE,
    num_train_steps=TRAIN_STEPS,
    num_frames=NUM_FRAMES,
)
LORA_DIR = train_lora(config)
print(f'\n✅ LoRA trained: {LORA_DIR}')

In [ ]:
#@title A2: Train on your own images (alternative to A1)
#@markdown Upload your reference images, then train.

import os
from pathlib import Path
from google.colab import files as colab_files
from src.config import ensure_dirs
from src.anecdote.lora_trainer import LoRATrainingConfig, train_lora

ensure_dirs()

LORA_NAME = 'my_style' #@param {type:"string"}
MODEL_SIZE = '14B' #@param ["14B", "1.3B"]
TRAIN_STEPS = 500 #@param {type:"integer"}

upload_dir = Path('/content/reference_data')
upload_dir.mkdir(exist_ok=True)

print('Upload your reference images (.png, .jpg) or videos (.mp4):')
uploaded = colab_files.upload()
for name, data in uploaded.items():
    (upload_dir / name).write_bytes(data)
    print(f'  Saved {name}')

print(f'\nTraining LoRA on {len(uploaded)} files...')
config = LoRATrainingConfig(
    reference_dir=upload_dir,
    lora_name=LORA_NAME,
    model_size=MODEL_SIZE,
    num_train_steps=TRAIN_STEPS,
)
LORA_DIR = train_lora(config)
print(f'\n✅ LoRA trained: {LORA_DIR}')

---
## Section B: Generate the anecdote video

Creates the AI intro for your demo:
1. SDXL generates keyframe images from your prompts
2. Wan2.1 animates them into video clips (using the LoRA if you trained one above)
3. FFmpeg concatenates into `anecdote.mp4`

In [ ]:
#@title B1: Configure prompts
#@markdown Edit these to match the project you're making a demo for.

keyframe_prompts = [
    'A developer sitting at a cluttered desk late at night, multiple browser tabs open on dual monitors, coffee cup empty, warm desk lamp lighting, photorealistic, cinematic shallow depth of field, 4K quality',
    'Close-up of a laptop screen showing a wall of error messages and red warning icons, reflection of a tired developer in the screen, dramatic lighting from the monitor, ultra detailed',
    'A developer throwing their hands up in frustration, papers and sticky notes scattered on a desk, dramatic side lighting, editorial photography style, 8K resolution',
]

motion_prompts = [
    'Camera slowly zooms in on the developer face, subtle head shake',
    'Screen content slowly scrolling down, flickering light from monitor',
    'Hands move upward in frustration, slight camera shake',
]

USE_MODEL_SIZE = '14B' #@param ["14B", "1.3B"]
USE_LORA = True #@param {type:"boolean"}

lora_path = LORA_DIR if (USE_LORA and 'LORA_DIR' in dir()) else None
print(f'Model: Wan2.1 {USE_MODEL_SIZE}')
print(f'LoRA:  {lora_path or "none (base model)"}')
print(f'Keyframes: {len(keyframe_prompts)}')

In [ ]:
#@title B2: Generate keyframe images (SDXL)
from src.anecdote.image_gen import generate_keyframes
from pathlib import Path

print('Generating keyframe images with SDXL...')
image_paths = generate_keyframes(keyframe_prompts, Path('/content/keyframes'))

print(f'\n✅ Generated {len(image_paths)} keyframes')

from IPython.display import display
from PIL import Image
for p in image_paths:
    display(Image.open(p).resize((640, 360)))

In [ ]:
#@title B3: Animate into video clips (Wan2.1 I2V + your LoRA)
from src.anecdote.video_gen import generate_video_from_images
from pathlib import Path

anecdote_path = Path('/content/anecdote.mp4')

print(f'Generating video clips with Wan2.1 {USE_MODEL_SIZE}...')
if lora_path:
    print(f'Using LoRA: {lora_path}')

generate_video_from_images(
    image_paths=image_paths,
    motion_prompts=motion_prompts,
    output_path=anecdote_path,
    lora_path=lora_path,
    model_size=USE_MODEL_SIZE,
)

if anecdote_path.exists():
    size_mb = anecdote_path.stat().st_size / 1024 / 1024
    print(f'\n✅ anecdote.mp4 ready ({size_mb:.1f} MB)')
else:
    print('⚠️  Generation failed — check the logs above')

---
## Section C: Download results to your Mac

In [ ]:
#@title C1: Direct download (through your browser)
from google.colab import files as colab_files
from pathlib import Path
import shutil

# Download the anecdote video
if Path('/content/anecdote.mp4').exists():
    print('Downloading anecdote.mp4...')
    colab_files.download('/content/anecdote.mp4')

# Download LoRA weights as a zip
if 'LORA_DIR' in dir() and Path(str(LORA_DIR)).exists():
    zip_name = f'/content/{Path(str(LORA_DIR)).name}_lora'
    shutil.make_archive(zip_name, 'zip', str(LORA_DIR))
    print(f'Downloading LoRA weights...')
    colab_files.download(f'{zip_name}.zip')

In [ ]:
#@title C2: Save to Google Drive
import shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

dest = Path('/content/drive/MyDrive/repovideo_results')
dest.mkdir(parents=True, exist_ok=True)

if Path('/content/anecdote.mp4').exists():
    shutil.copy2('/content/anecdote.mp4', dest / 'anecdote.mp4')
    print(f'✅ anecdote.mp4 → Google Drive')

if 'LORA_DIR' in dir() and Path(str(LORA_DIR)).exists():
    lora_dest = dest / 'loras' / Path(str(LORA_DIR)).name
    shutil.copytree(str(LORA_DIR), str(lora_dest), dirs_exist_ok=True)
    print(f'✅ LoRA weights → Google Drive')

print(f'\nAll saved to: {dest}')
print('Files will sync to your Mac via Google Drive for Desktop.')

---
## Using the results on your Mac

```bash
# Option 1: Just plug in the anecdote video you downloaded
repovideo generate https://github.com/user/repo \
    --anecdote-file ~/Downloads/anecdote.mp4

# Option 2: Install the LoRA locally for future runs
unzip ~/Downloads/developer_lora.zip -d ~/.repovideo/loras/developer/
repovideo generate https://github.com/user/repo \
    --lora-name developer --model-size 1.3B

# Option 3: Full automation via Google Drive
# (run Section D in Colab, then on your Mac:)
repovideo generate https://github.com/user/repo --colab
```

---
## Section D: Automated job watcher (for `--colab` flag)

Leave this running in Colab. On your Mac, use `repovideo generate <url> --colab`
and the GPU work gets dispatched here automatically via Google Drive.

In [ ]:
#@title D1: Start job watcher
import json
import shutil
import subprocess
import time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from src.config import ensure_dirs
from src.anecdote.image_gen import generate_keyframes
from src.anecdote.video_gen import generate_video_from_images
from src.anecdote.lora_trainer import LoRATrainingConfig, train_lora

ensure_dirs()
JOBS_DIR = Path('/content/drive/MyDrive/repovideo_jobs')
JOBS_DIR.mkdir(parents=True, exist_ok=True)

def process_job(job_dir):
    job_dir = Path(job_dir)
    config = json.loads((job_dir / 'config.json').read_text())
    results_dir = job_dir / 'results'
    results_dir.mkdir(exist_ok=True)
    job_type = config['job_type']
    print(f'  Processing: {job_type}')

    if job_type == 'generate_anecdote':
        kf_dir = results_dir / 'keyframes'
        img_paths = generate_keyframes(config['keyframe_prompts'], kf_dir)
        anecdote_path = results_dir / 'anecdote.mp4'
        generate_video_from_images(
            img_paths, config['motion_prompts'], anecdote_path,
            lora_path=Path(config['lora_path']) if config.get('lora_path') else None,
            model_size=config.get('model_size', '14B'),
        )

    elif job_type == 'train_lora':
        tc = LoRATrainingConfig(
            reference_dir=job_dir / 'reference_data',
            lora_name=config.get('lora_name', 'custom_style'),
            model_size=config.get('model_size', '14B'),
            num_train_steps=config.get('num_train_steps', 500),
        )
        lora_dir = train_lora(tc)
        shutil.copytree(str(lora_dir), str(results_dir / 'lora'), dirs_exist_ok=True)

    (job_dir / 'status.txt').write_text('completed')
    print(f'  ✅ Done')

print(f'Watching {JOBS_DIR} for jobs...')
print('On your Mac: repovideo generate <url> --colab\n')

while True:
    for jd in sorted(JOBS_DIR.iterdir()):
        if not jd.is_dir(): continue
        sf = jd / 'status.txt'
        if not sf.exists() or sf.read_text().strip() != 'pending': continue
        try:
            process_job(jd)
        except Exception as e:
            print(f'  ❌ Failed: {e}')
            (jd / 'status.txt').write_text(f'failed: {e}')
    time.sleep(15)